In [2]:

import os
import requests

from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain.agents import create_agent


# 1. Get API keys securely using getpass

GOOGLE_API_KEY = getpass("Enter Gemini API Key: ")
TAVILY_API_KEY = getpass("Enter Tavily API Key: ")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY


# 2. Initialize Gemini LLM

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    google_api_key=GOOGLE_API_KEY
)


# 3. Create Simple QA Tool

qa_prompt = PromptTemplate.from_template(
    "Answer clearly and accurately: {question}"
)

def run_qa(question):
    qa_chain = qa_prompt | llm

    response = qa_chain.invoke({
        "question": question
    })

    return response.content


qa_tool = Tool(
    name="Simple_QA",
    func=run_qa,
    description=(
        "Answers general factual and conceptual questions. "
        "Use this tool for questions that do not require current web information."
    )
)


# 4. Create Tavily Web Search Tool

def run_web_search(query):

    url = "https://api.tavily.com/search"

    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "max_results": 3
    }

    response = requests.post(
        url,
        json=payload,
        timeout=30
    )

    response.raise_for_status()

    results = response.json().get("results", [])

    if not results:
        return "No search results found."

    formatted_results = []

    for result in results:
        formatted_results.append(
            f"Title: {result.get('title', '')}\n"
            f"URL: {result.get('url', '')}\n"
            f"Content: {result.get('content', '')}"
        )

    return "\n\n".join(formatted_results)


search_tool = Tool(
    name="Web_Search",
    func=run_web_search,
    description=(
        "Searches the internet for current information, "
        "latest news, recent updates, and current events."
    )
)


# 5. Combine Tools

tools = [
    qa_tool,
    search_tool
]


# 6. Create Conversational AI Agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a helpful AI assistant with access to two tools.

    1. Simple_QA:
       Answers general factual and conceptual questions.

    2. Web_Search:
       Searches the internet for current information.

    Instructions:

    - Understand the user's query.
    - Select the most appropriate tool.
    - Use Simple_QA for general knowledge questions.
    - Use Web_Search for latest or current information.
    - Use previous conversation context when relevant.
    - Use tool results to generate a clear final answer.
    - Do not claim to have searched the web unless Web_Search was used.
    """
)


# 7. Maintain Conversation History

conversation_history = []

while True:

    query = input("\nYou: ")

    if query.lower().strip() in ["exit", "quit", "bye"]:
        print("\nAssistant: Goodbye!")
        break

    try:

        response = agent.invoke({
            "messages": conversation_history + [
                {"role": "user", "content": query}
            ]
        })

        conversation_history = response["messages"]

        print("\nAssistant:")
        print(response["messages"][-1].content)

    except Exception as e:
        print("\nError:", str(e))


Enter Gemini API Key: ··········
Enter Tavily API Key: ··········

You: what is langchain

Assistant:
[{'type': 'text', 'text': '**LangChain** is an open-source software framework designed to help developers build applications powered by Large Language Models (LLMs) such as OpenAI\'s GPT-4, Anthropic\'s Claude, or open-source models like Llama.\n\nIn simple terms, **LangChain acts as the "glue"** that connects a language model to external data sources, tools, and custom logic, allowing developers to create complex, intelligent AI applications.\n\n---\n\n### Why is LangChain used?\n\nBy itself, a language model is primarily a text-in/text-out generator relying on pre-trained knowledge. LangChain overcomes these limitations by enabling LLMs to:\n\n1. **Access Private/External Data (RAG):** Connect models to PDFs, databases, APIs, or internal documents so the AI can answer questions using specific, up-to-date data (*Retrieval-Augmented Generation*).\n2. **Take Actions (Agents):** Allow th

In [3]:
%pip install -q langchain-tavily --no-deps

In [6]:
%pip install -q --no-deps "langchain-google-genai==4.4.3"

In [9]:
%pip install -q filetype

In [3]:
from langchain.agents import create_agent


agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a helpful conversational AI assistant.

    Instructions:
    - Answer user questions clearly.
    - Use available tools whenever appropriate.
    - Maintain conversation context.
    - Use tool results to generate accurate answers.
    """
)


conversation_history = []

query = "Summarize LangChain in 2 lines, then tell me who created it."

response = agent.invoke({
    "messages": conversation_history + [
        {"role": "user", "content": query}
    ]
})

conversation_history = response["messages"]

print("Agent Response:")
print(response["messages"][-1].content)

Agent Response:
[{'type': 'text', 'text': '**Summary:**  \nLangChain is an open-source framework designed to simplify the development of applications powered by large language models (LLMs).  \nIt provides tools to chain together prompts, external data, memory, and APIs, enabling developers to build context-aware systems and autonomous AI agents.\n\n**Creator:**  \nLangChain was created by **Harrison Chase** (launched in October 2022).', 'extras': {'signature': 'Es8JCswJAWkUfRPozsfdDPjXZa0MCtUfkZOYtjHa1FAs5qEA2MLwYnpWfxlWsgolN5D3NzPw62COvrY+giAj9XmqsuHKWGtdIGYs3DtXaNJRcL2signyOBOuYsQ7Na8OOWn8kxRjgz0TXrGVToIZCgLoLM7/Wgxm8iJ1n8fCCECDsfw5ITwPcF7Q8cDqABP9ThPo/08gJVSoCyJdmfi8Uewz0jrWGBiV355O1QMKST0GbuvYkIgUnuDY3CWyokamPnA0Y7Qv18rxoLe26elb2UUR+15ho5C0aMepSxhIrNh4I8UD9s/WeBxyGg1APi2TjgDkZ93j5O5dXzyozuRj3EnESWDIObiWmL05PZdKuuhEHqGjrCLFvMxDAkR3Na4sanMN81DHCuf/jEW3YHvdscF22LXlRrRvAi85V42g5ZVqjvx/J7zDaXimBdY0wKiLzYQ0MyVS7G2zRonCVW6FrOmfoxFQ43PwkoZY2TQfVhFivVT0xeXZwK7cofQqs++KFxqFoGGOy4y4A+IbcDC8a

In [4]:

from langchain.agents import create_agent


agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a helpful conversational AI assistant.

    Instructions:
    - Understand the user's request.
    - Use available tools when necessary.
    - Maintain conversation context.
    - Provide clear, structured and accurate responses.
    - For study plans, provide practical step-by-step guidance.
    """
)

conversation_history = []

query = "Plan a 3-step study path for LangChain."

response = agent.invoke({
    "messages": conversation_history + [
        {"role": "user", "content": query}
    ]
})

conversation_history = response["messages"]

print("Agent Response:")
print(response["messages"][-1].content)





Agent Response:
[{'type': 'text', 'text': 'Here is a structured **3-step study path** designed to take you from the fundamentals of LangChain to building advanced, production-ready AI applications.\n\n---\n\n### **Prerequisites**\n* Intermediate Python knowledge (async code, functions, type hinting).\n* Basic understanding of LLMs (prompts, tokens, temperature) and API keys (e.g., OpenAI, Anthropic, or local LLMs via Ollama).\n\n---\n\n### **Step 1: Core Fundamentals & LCEL**\n**Goal:** Master the primary abstractions of LangChain and learn how to compose workflows using **LangChain Expression Language (LCEL)**.\n\n* **Key Concepts:**\n  * **Chat Models:** Interacting with LLMs (`ChatOpenAI`, `ChatAnthropic`, HuggingFace, Ollama).\n  * **Prompt Templates:** Dynamically formatting user prompts using `ChatPromptTemplate`.\n  * **Output Parsers:** Converting unstructured LLM output into structured data (JSON, Pydantic models, lists).\n  * **LCEL (LangChain Expression Language):** Combinin

In [5]:

from langchain_core.tools import StructuredTool
from langchain.agents import create_agent
from pydantic import BaseModel, Field


# 1. Define the input schema using Pydantic

class TitleInput(BaseModel):
    topic: str = Field(description="The topic for the title")
    tone: str = Field(
        default="concise",
        description="The tone of the title, such as friendly or professional"
    )


# 2. Define the tool function

def title_tool_fn(topic: str, tone: str = "concise") -> str:
    return f"{tone.title()} Title: {topic} in Practice"


# 3. Create the structured tool

title_tool = StructuredTool.from_function(
    name="TitleMaker",
    func=title_tool_fn,
    description="Generate a title given a topic and an optional tone.",
    args_schema=TitleInput
)


# 4. Initialize the agent using Gemini

agent = create_agent(
    model=llm,
    tools=[title_tool],
    system_prompt="""
    You are a helpful AI assistant.

    Use the TitleMaker tool whenever the user requests a title.
    Understand the requested topic and tone before generating the title.
    """
)


# 5. Run the agent

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Make a friendly title about LangGraph tutorials."
        }
    ]
})


print("Result:", response["messages"][-1].content)


Result: [{'type': 'text', 'text': 'Here is a friendly title for your topic:\n\n**LangGraph Tutorials in Practice**', 'extras': {'signature': 'EocCCoQCAWkUfRNo0EFzeyd2qtxFNN87GN9Ifconqj42FD1wCUoW0ikHwPi5nWVdkb7UC2Ju8LmqABQHw/kx73QfYZQkc5vOfhC08bq6RvnOKsW2v+FWUW1O/OnYSgtQShQMKRpKRzpEZii9HCHUwrCvyfGiqd1g3pk5SA6T0Q2h8J0gzNFkhXrpPMTSCR6XUyKjWtAQEEk74TNHirzjHdnfgJALhHu6RppLTURZ5j8p8hHb3W+AjuDa+dcDfY6xvwKaL1LcKx2AhP7sMBtH9uWqGPwcG27eVlx111vMXEsO7CKIEEe1FUT9ZSBxCLW7VVTRpCA6FKpNyKr+0U/HC3QncmRRwDBbxhc='}}]


In [8]:

import os
import requests

from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import Tool
from langchain.agents import create_agent


# 1. Get API keys securely using getpass

GOOGLE_API_KEY = getpass("Enter Gemini API Key: ")
TAVILY_API_KEY = getpass("Enter Tavily API Key: ")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY


# 2. Initialize Gemini LLM

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    google_api_key=GOOGLE_API_KEY
)


# 3. Define Tavily Search Function

def run_web_search(query):

    url = "https://api.tavily.com/search"

    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "max_results": 1
    }

    response = requests.post(
        url,
        json=payload,
        timeout=30
    )

    response.raise_for_status()

    results = response.json().get("results", [])

    if not results:
        return "No search results found."

    formatted_results = []

    for result in results:

        formatted_results.append(
            f"Title: {result.get('title', '')}\n"
            f"URL: {result.get('url', '')}\n"
            f"Content: {result.get('content', '')}"
        )

    return "\n\n".join(formatted_results)


# 4. Create Intermediate Answer Search Tool

search_tool = Tool(
    name="Intermediate_Answer",
    func=run_web_search,
    description=(
        "Search the web for missing facts and intermediate answers. "
        "Use this tool to search for individual facts needed "
        "to answer a larger question."
    )
)


# 5. Create Self-Ask Style Agent

agent = create_agent(
    model=llm,
    tools=[search_tool],
    system_prompt="""
    You are a Self-Ask with Search assistant.

    Instructions:

    - Understand the user's question.
    - Break complex questions into smaller sub-questions.
    - Identify missing facts.
    - Use the Intermediate_Answer tool to search for those facts.
    - Search for each important missing fact when necessary.
    - Combine the search results.
    - Provide a clear and accurate final answer.
    - Do not invent facts that are not supported by search results.
    """
)


# 6. Run the Agent

query = "When was the first LangChain release and who founded it?"

try:

    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": query
            }
        ]
    })

    print("\nUser:", query)

    print("\nAssistant:")
    print(result["messages"][-1].content)

except Exception as e:

    print("\nError:", str(e))


Enter Gemini API Key: ··········
Enter Tavily API Key: ··········

User: When was the first LangChain release and who founded it?

Assistant:
[{'type': 'text', 'text': "Are there any specific sub-questions needed?\nFollow up: Who founded LangChain and when was its first release?\n\nLet's search via `Intermediate_Answer`.`default_api:Intermediate_Answer`{__arg1:"}, {'type': 'text', 'text': 'who founded LangChain and initial release date'}, {'type': 'text', 'text': '}'}, {'type': 'text', 'text': '', 'extras': {'signature': 'ErEDCq4DAWkUfRPj+aYoOY0IvkY07lNX+orfoETmcCpiCKUKTdWZyVTkxO2Zew7XcneYMgbK+GDG0GnfTUac7FOgN6BW/0aCIgXljmnwJH8T1plTQwtohfeLZFv9qgHNfSEveRqERou+IeA97xpjs0tki/KU/q7vgq5YUkDRzdNEHC655vTUNu4HEUJXJw3fJn+LZC04nZpAUom2DaLYIwRJV+53uCZhc2CX9WjmNI4hMt44YM39RduLhMSO0OPmw63BSe0r7FbpfcHBha6E5Ch4GTOXgmyXiLWITD8p2tKKDTCuSLZ3hYYzSHi+UWRXnFJmu6GIv/8UXVHOvJft2PNr7kq8/LnqwgsOy7EswlJEj3aZ4QpLHnHU8msCdeQe3DaXY974u1Vds+ZKxPkqf84I9hVa28XADfTGkRDcFAzvuDfxjaHPNaWRM5qttGUo6Q7Ooo8WSOQ7oepYn6qsXMo8